In [4]:
import pandas as pd
df = pd.read_csv('everpeak_retail.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5008 entries, 0 to 5007
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          5008 non-null   int64  
 1   order_date        5000 non-null   object 
 2   customer_id       5008 non-null   int64  
 3   product_category  5008 non-null   object 
 4   price             5008 non-null   int64  
 5   quantity          5008 non-null   int64  
 6   order_value       5008 non-null   int64  
 7   payment_method    5008 non-null   object 
 8   city              4908 non-null   object 
 9   state             4908 non-null   object 
 10  customer_age      4858 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 430.5+ KB


In [5]:
def reemplazar_sentinels_global(df, numeric_cols, text_cols):
    numeric_sentinels = [-999, 999]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].replace(numeric_sentinels, pd.NA)
    text_sentinels = ["?"]
    for col in text_cols:
        df[col] = df[col].replace(text_sentinels, "unknown")
    return df

def crear_flags(df, flags_cols):
    for col in flags_cols:
        nombre_flag = col + "_missing_flag"
        df[nombre_flag] = df[col].isna().astype(int)
    return df

def imputar_segun_diagnostico(df, median_fill_cols, fill_unknown_cols, date_drop_cols):
    for col in median_fill_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        med = df[col].median()
        df[col] = df[col].fillna(med)
    for col in fill_unknown_cols:
        df[col] = df[col].fillna("unknown")
    for col in date_drop_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df = df.dropna(subset=[col]).reset_index(drop=True)
    return df

def clean_data(df, numeric_cols, text_cols, flags_cols,
               median_fill_cols, fill_unknown_cols, date_drop_cols):
    df = reemplazar_sentinels_global(df, numeric_cols, text_cols)
    df = crear_flags(df, flags_cols)
    df = imputar_segun_diagnostico(df, median_fill_cols, fill_unknown_cols, date_drop_cols)
    return df

columnas_numericas = ["customer_age"]
columnas_texto = ["product_category"]
columnas_flags = ["customer_age", "city", "state"]
cols_imputar_mediana = ["customer_age"]
cols_imputar_unknown = ["city", "state"]
cols_imputar_fecha = ["order_date"]

df = clean_data(df, columnas_numericas, columnas_texto, columnas_flags,
                cols_imputar_mediana, cols_imputar_unknown, cols_imputar_fecha)

df.info()
print(df.head())
df.to_csv("everpeak_clean.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   order_id                   5000 non-null   int64         
 1   order_date                 5000 non-null   datetime64[ns]
 2   customer_id                5000 non-null   int64         
 3   product_category           5000 non-null   object        
 4   price                      5000 non-null   int64         
 5   quantity                   5000 non-null   int64         
 6   order_value                5000 non-null   int64         
 7   payment_method             5000 non-null   object        
 8   city                       5000 non-null   object        
 9   state                      5000 non-null   object        
 10  customer_age               5000 non-null   float64       
 11  customer_age_missing_flag  5000 non-null   int64         
 12  city_m

In [7]:
from google.colab import files
df.to_csv("everpeak_clean.csv", index=False)
files.download("everpeak_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>